Imagine you're working as a Data Engineer for an e-commerce company.

The source system sends you an orders dataset every day.

Unfortunately, the source data is dirty.

Create DataFrame

In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Day 2").getOrCreate()
print(spark)

In [0]:
data = [
    ("O101", " C001 ", "Laptop", "1500.50", 2, "2026-08-01"),
    ("O102", "C002", "Mobile", "800", None, "2026-08-01"),
    ("O103", " C003", "Tablet", "-500", 1, "2026-08-02"),
    ("O104", "C004 ", "Laptop", "1200.75", 1, "invalid_date"),
    ("O105", "C005", "Mobile", "900", 2, "2026-08-03"),
    ("O105", "C005", "Mobile", "900", 2, "2026-08-03"),
    ("O106", None, "Tablet", "700", 1, "2026-08-03"),
]

columns = [
    "order_id",
    "customer_id",
    "product",
    "amount",
    "quantity",
    "order_date"
]

df=spark.createDataFrame(data, columns)
df.show()
df.printSchema()

##### The business team tells you:
#####
###### We cannot directly use this data for reporting. Clean the dataset and create a reliable orders table.

Problem 1 — Customer IDs

Some `customer_id `values contain unnecessary spaces.

In [0]:
from pyspark.sql.functions import *

In [0]:
df=df.withColumn('customer_id',trim(col('customer_id')))
df.show()

Problem 2 — Amount

Currently:

`amount
- 1500.50
- 800
- -500`

The schema will show that amount is probably a string.

Requirement

Convert amount into a proper numeric datatype.

Question: Which datatype would you choose for a money column?

In [0]:
df=df.withColumn('amount',col('amount').cast('float'))
df.show()
df.printSchema()

Problem 3 — Missing Quantity

One order has:

`quantity = NULL`

Business rule:

If quantity is missing, assume quantity = 1.

Requirement

Replace NULL quantity with 1.

Write the code.

In [0]:
from pyspark.sql.functions import when, col, lit
df=df.withColumn('quantity', when(col('quantity').isNull(), lit(1)).otherwise(col('quantity')))
df.show()

Problem 4 — Invalid Amount

There is an order:

amount = -500

Business rule:

An order amount cannot be negative.

We don't want to delete the order yet.

Instead, create a new column:

amount_status

In [0]:
df=df.withColumn('amount_status',when(col('amount')<0,lit('Invalid') ).otherwise(lit('Valid')))

df.show()

In [0]:
df.show()

Problem 5 — Order Date

One record contains:

invalid_date

The business wants order_date as a proper date.

Requirement

Convert order_date from string → date.

But be careful:

Invalid dates should become NULL rather than crashing the pipeline.

In [0]:
from pyspark.sql.functions import try_to_date

In [0]:
df=df.withColumn('order_date',try_to_date(col('order_date'),'yyyy-MM-dd'))
df.show()

Problem 6 — Duplicate Orders

You notice:

O105
O105

The same order appears twice.

Business rule:

order_id should uniquely identify an order.

Requirement

Remove duplicate orders.

In [0]:
df=df.dropDuplicates()
df.show()
df.printSchema()

Problem 7 — Missing Customer

One order has:

customer_id = NULL

Business rule:

Orders without a customer cannot be used for customer-level reporting.

Remove those records.

In [0]:
df=df.dropna(subset=['customer_id'])
df.show()

Final Business Requirement

After cleaning, create a new column:

order_value_category

Rules:

amount >= 1000       → "HIGH"
amount >= 500        → "MEDIUM"
amount < 500         → "LOW"

For example:

amount	category
1500.50	HIGH
800	MEDIUM
700	MEDIUM
1200.75	HIGH

But remember our -500 record is invalid.

So think about whether you should classify it before or after handling invalid amounts.

In [0]:
df=df.withColumn('order_value_category',when(col('amount_status')=='Valid',\
    when(col('amount')>=1000, lit('High')).\
        when(col('amount')>=50, lit('Medium')).\
            otherwise('Low')).\
                otherwise('Invalid'))


df.show()